# This notebook fit a GEV to maximum annual percipitation using MLE and bayesian methods
##### Author: Omid Emamjomehzadeh (https://www.omidemam.com/)
##### Supervisor: Dr. Omar Wani (https://engineering.nyu.edu/faculty/omar-wani)
##### Hydrologic Systems Group @NYU (https://www.omarwani.com/)

In [11]:
#import libraries
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib as mpl
import pandas as pd
import numpy as np
import pymc3 as pm3
import scipy.stats as stats
from scipy import stats
from scipy.stats import genextreme
from scipy.stats import norm, gaussian_kde
from scipy.optimize import minimize
import seaborn as sns
import arviz as az
import pymc3 as pm
import os
import theano.tensor as tt
from theano.compile.ops import as_op
import theano
import pandas as pd
from pymc3.distributions.dist_math import bound
from scipy.stats import genextreme
import warnings
from arviz.plots import plot_utils as azpu
import arviz as az
from tqdm import tqdm
%matplotlib inline
sns.set()
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore', UserWarning)
theano.config.warn.round=False
from watermark import watermark
from datetime import datetime
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import random


# Read the data

In [2]:
# --- Paths ---
BASE_DIR = r"D:\BMM-IDF4Drainage_data_results\Data\AORC"
CITIES_CSV = r"D:\BMM-IDF4Drainage_data_results\us_cities_coordinates.csv"

# --- Load city list ---
cities_df = pd.read_csv(CITIES_CSV)
city_names = cities_df["City"].astype(str).str.strip().tolist()

# --- Final container ---
dataframes = {}

# --- Helper to read a CSV safely ---
def read_if_exists(path, usecols=None):
    if os.path.exists(path):
        return pd.read_csv(path, usecols=usecols)
    return None

# --- Build combined structure ---
for city_full in city_names:
    # Strip the state to get short name
    city_short = city_full.split(",")[0].strip()

    # Paths to the hourly and daily data
    hourly_path = os.path.join(BASE_DIR, city_full, f"{city_full}_annual_max_hourly.csv")
    daily_path  = os.path.join(BASE_DIR, city_full, f"{city_full}_annual_max_daily.csv")

    df_hourly = read_if_exists(hourly_path, usecols=["year", "max_hourly_mm"])
    df_daily  = read_if_exists(daily_path,  usecols=["year", "max_daily_mm"])

    if df_hourly is not None and df_daily is not None:
        # Keep only the common years between both
        df = pd.merge(df_hourly, df_daily, on="year", how="inner")

        # Rename columns to match your original structure
        df.rename(columns={"max_hourly_mm": "1h", "max_daily_mm": "24h"}, inplace=True)

        # Filter to 1979–2024 (if desired)
        df = df[df["year"].between(1979, 2024)]

        # Sort by year and reset index
        df = df.sort_values("year").reset_index(drop=True)

        # Save to dict with short name
        dataframes[city_short] = df

# --- print ---
print(dataframes["Birmingham"].head())
print(dataframes["Phoenix"])
print(list(dataframes.keys()))
# cities names
cities = ['Birmingham', 'Phoenix', 'Little Rock', 'Los Angeles', 'Denver', 'Hartford', 'Wilmington', 'Miami', 'Atlanta', 'Boise',
           'Chicago', 'Indianapolis', 'Des Moines', 'Wichita', 'Louisville', 'New Orleans', 'Portland', 'Baltimore', 'Boston', 'Detroit',
             'Minneapolis', 'Jackson', 'St. Louis', 'Billings', 'Omaha', 'Las Vegas', 'Manchester', 'Newark', 'Albuquerque', 'New York City',
               'Charlotte', 'Fargo', 'Columbus', 'Oklahoma City', 'Philadelphia', 'Providence', 'Charleston', 'Sioux Falls', 'Nashville', 'Houston',
                 'Salt Lake City', 'Burlington', 'Richmond', 'Seattle', 'Milwaukee', 'Cheyenne']             



   year         1h         24h
0  1979  21.200000  135.500002
1  1980  18.300000  102.500002
2  1981  43.200001   77.800001
3  1982  35.500001   74.100001
4  1983  24.100000  103.300002
    year         1h        24h
0   1979   8.400000  31.300000
1   1980   3.500000  14.100000
2   1981   4.600000  22.600000
3   1982   6.800000  24.700000
4   1983   9.700000  40.700001
5   1984   8.400000  32.300000
6   1985  10.400000  27.200000
7   1986   5.700000  28.900000
8   1987  15.000000  34.300001
9   1988   4.700000  26.100000
10  1989  20.400000  29.400000
11  1990   7.900000  29.400000
12  1991  20.200000  25.200000
13  1992  13.500000  42.400001
14  1993   7.700000  35.000001
15  1994  10.800000  15.300000
16  1995   6.800000  21.200000
17  1996   5.900000  11.300000
18  1997  11.300000  17.600000
19  1998   9.500000  18.200000
20  1999  11.900000  26.600000
21  2000   8.300000  51.100001
22  2001   6.000000  16.800000
23  2002   7.200000  13.700000
24  2003  16.100000  32.700000
25  2004

# Read CMIP6 data (2015-2100)

In [ ]:
# --- Paths ---
BASE_DIR = r"D:\BMM-IDF4Drainage_data_results\Data\CMIP6"
CITIES_CSV = r"D:\BMM-IDF4Drainage_data_results\us_cities_coordinates.csv"

# --- Load city list ---
cities_df = pd.read_csv(CITIES_CSV)
city_names = cities_df["City"].astype(str).str.strip().tolist()

# --- Final container ---
dataframes_ssp = {}

# --- Helper to read a CSV safely ---
def read_if_exists(path, usecols=None):
    if os.path.exists(path):
        return pd.read_csv(path, usecols=usecols)
    return None

# --- Build combined structure ---
for city_full in city_names:
    # Strip the state to get short name
    city_short = city_full.split(",")[0].strip()

    # Paths to the hourly and daily data
    daily_path_126  = os.path.join(BASE_DIR, city_full, f"{city_full}_SSP126_yearly_max.csv")
    daily_path_370  = os.path.join(BASE_DIR, city_full, f"{city_full}_SSP370_yearly_max.csv")
    daily_path_585  = os.path.join(BASE_DIR, city_full, f"{city_full}_SSP585_yearly_max.csv")


    df_daily_126  = read_if_exists(daily_path_126,  usecols=["year", "max_daily_mm"])
    df_daily_370  = read_if_exists(daily_path_370,  usecols=["year", "max_daily_mm"])
    df_daily_585  = read_if_exists(daily_path_585,  usecols=["year", "max_daily_mm"])

    if (df_daily_126 is not None) and (df_daily_370 is not None) and (df_daily_585 is not None):

        # rename columns so they don't collide, then merge pairwise
        df_daily_126 = df_daily_126.rename(columns={"max_daily_mm": "126"})
        df_daily_370 = df_daily_370.rename(columns={"max_daily_mm": "370"})
        df_daily_585 = df_daily_585.rename(columns={"max_daily_mm": "585"})

        # Merge two at a time
        df = df_daily_126.merge(df_daily_370, on="year", how="inner").merge(df_daily_585, on="year", how="inner")

        # Optional filter range (CMIP6 futures typically 2015–2100)
        df = df[df["year"].between(2015, 2100)]

        df = df.sort_values("year").reset_index(drop=True)
        # --- Split each scenario into two new columns ---
        df["126_2025_2100"] = np.where(df["year"].between(2025, 2100), df["126"], np.nan)
        df["126_2025_2064"] = np.where(df["year"].between(2025, 2064), df["126"], np.nan)
        df["126_2065_2100"] = np.where(df["year"].between(2065, 2100), df["126"], np.nan)

        df["370_2025_2100"] = np.where(df["year"].between(2025, 2100), df["370"], np.nan)
        df["370_2025_2064"] = np.where(df["year"].between(2025, 2064), df["370"], np.nan)
        df["370_2065_2100"] = np.where(df["year"].between(2065, 2100), df["370"], np.nan)

        df["585_2025_2100"] = np.where(df["year"].between(2025, 2100), df["585"], np.nan)
        df["585_2025_2064"] = np.where(df["year"].between(2025, 2064), df["585"], np.nan)
        df["585_2065_2100"] = np.where(df["year"].between(2065, 2100), df["585"], np.nan)
        df["585_2075_2100"] = np.where(df["year"].between(2075, 2100), df["585"], np.nan)

        dataframes_ssp[city_short] = df

# --- print ---
print(dataframes_ssp["Birmingham"].head())
print(dataframes_ssp["Phoenix"])
print(list(dataframes_ssp.keys()))
# cities names
cities = ['Birmingham', 'Phoenix', 'Little Rock', 'Los Angeles', 'Denver', 'Hartford', 'Wilmington', 'Miami', 'Atlanta', 'Boise',
           'Chicago', 'Indianapolis', 'Des Moines', 'Wichita', 'Louisville', 'New Orleans', 'Portland', 'Baltimore', 'Boston', 'Detroit',
             'Minneapolis', 'Jackson', 'St. Louis', 'Billings', 'Omaha', 'Las Vegas', 'Manchester', 'Newark', 'Albuquerque', 'New York City',
               'Charlotte', 'Fargo', 'Columbus', 'Oklahoma City', 'Philadelphia', 'Providence', 'Charleston', 'Sioux Falls', 'Nashville', 'Houston',
                 'Salt Lake City', 'Burlington', 'Richmond', 'Seattle', 'Milwaukee', 'Cheyenne']             



# Fit the parameters based on Maximum Likelihood Estimation (1hr)

In [3]:
# Define GEV log-likelihood function
def gev_loglik(theta, data):
    c, loc, scale = theta
    gev = genextreme(c=c, loc=loc, scale=scale)
    return -np.sum(np.log(gev.pdf(data)))

# Load dataframes for each city

sample_number = 20000

T = np.array([2, 5, 10, 25, 50, 100])
p = 1 - 1 / T

# Initialize arrays to store results
save_samples_mle = np.zeros([len(cities),3])
p_mle_all_cities = np.zeros([len(cities), sample_number])
percentiles_all_cities = np.zeros([len(cities), 6])
percentiles = [50, 80, 90, 96, 98, 99]

# Iterate over all cities
for i in range(len(cities)):
    # Get 24h data
    data24h = np.array(dataframes.get(cities[i])["1h"])
    
    # Remove NaN values
    data24h = data24h[~np.isnan(data24h)]
    
    # Fit GEV distribution using MLE
    res = minimize(gev_loglik, x0=[0, 0, 1], args=(data24h,), method='Nelder-Mead')
    c_mle, loc_mle, scale_mle = res.x
    save_samples_mle[i, 0] = c_mle
    save_samples_mle[i, 1] = loc_mle
    save_samples_mle[i, 2] = scale_mle
    
    # Generate samples using MLE parameters
    #p_mle_all_cities[i, :] = genextreme.rvs(c=c_mle, loc=loc_mle, scale=scale_mle, size=sample_number)
    
    # Calculate percentiles
    #percentiles_all_cities[i, :] = np.percentile(p_mle_all_cities[i, :], percentiles)
    percentiles_all_cities[i, :] = genextreme.ppf(p, c_mle, loc_mle, scale_mle)

# Save results to files
#np.savez(r"D:\IDF bayesian\samples\all_samples_and_p_mle_24h_AORC.npz", saved_samples_24h_mle=save_samples_mle, p_mle_24h=p_mle_all_cities)
percentile_table = pd.DataFrame(percentiles_all_cities, columns=percentiles, index=cities)
percentile_table.index.name = 'City'
percentile_table.to_csv(r'D:\BMM-IDF4Drainage_data_results\Percentile\percentile_table_1h_mle_AORC.csv')

params_table = pd.DataFrame(save_samples_mle, columns=['shape(c)', 'loc', 'scale'], index=cities)
params_table.index.name = 'City'
params_table.to_csv(r'D:\BMM-IDF4Drainage_data_results\Percentile\gev_params_1h_mle_AORC.csv')

In [4]:
# Load the csv table as a DataFrame
df = pd.read_csv(r'D:\BMM-IDF4Drainage_data_results\Percentile\percentile_table_1h_mle_AORC.csv')
df = df.round(1)
# Create a dictionary to map old column names to new column names
col_mapping = {'50': '2', '80': '5', '90': '10', '96': '25', '98': '50', '99': '100'}
# Rename columns using the dictionary
df = df.rename(columns=col_mapping)
# Save the renamed CSV file
df.to_csv(r'D:\BMM-IDF4Drainage_data_results\Percentile\return_levels(mm)_1h_mle_AORC.csv', index=False)
print(df)


              City     2     5    10    25    50    100
0       Birmingham  26.2  37.0  43.7  51.7  57.2   62.5
1          Phoenix   9.6  14.9  19.5  26.7  33.4   41.4
2      Little Rock  25.4  36.5  44.2  54.4  62.3   70.5
3      Los Angeles  13.9  20.1  24.1  28.9  32.3   35.7
4           Denver   9.0  16.3  23.4  36.2  49.5   67.2
5         Hartford  16.8  26.3  34.5  47.7  60.1   75.2
6       Wilmington  21.5  31.0  37.5  46.0  52.5   59.1
7            Miami  23.3  38.2  51.0  71.7  91.1  114.6
8          Atlanta  19.4  29.5  37.8  50.4  61.7   74.9
9            Boise   6.4   9.4  11.6  15.0  17.8   21.1
10         Chicago  18.1  24.5  28.2  32.1  34.7   36.9
11    Indianapolis  19.3  28.0  34.7  44.3  52.4   61.3
12      Des Moines  19.6  29.2  36.8  48.3  58.2   69.6
13         Wichita  22.8  34.9  43.6  55.4  64.9   74.8
14      Louisville  20.9  31.0  38.2  47.8  55.3   63.2
15     New Orleans  34.2  49.1  59.0  71.4  80.7   89.8
16        Portland   8.6  11.8  14.3  18.1  21.4

# Maximum Likelihood Estimation (24h)

## AORC

In [5]:
# Define GEV log-likelihood function
def gev_loglik(theta, data):
    c, loc, scale = theta
    gev = genextreme(c=c, loc=loc, scale=scale)
    return -np.sum(np.log(gev.pdf(data)))

# Load dataframes for each city

sample_number = 20000

T = np.array([2, 5, 10, 25, 50, 100])
p = 1 - 1 / T

# Initialize arrays to store results
save_samples_mle = np.zeros([len(cities),3])
p_mle_all_cities = np.zeros([len(cities), sample_number])
percentiles_all_cities = np.zeros([len(cities), 6])
percentiles = [50, 80, 90, 96, 98, 99]

# Iterate over all cities
for i in range(len(cities)):
    # Get 24h data
    data24h = np.array(dataframes.get(cities[i])["24h"])
    
    # Remove NaN values
    data24h = data24h[~np.isnan(data24h)]
    
    # Fit GEV distribution using MLE
    res = minimize(gev_loglik, x0=[0, 0, 1], args=(data24h,), method='Nelder-Mead')
    c_mle, loc_mle, scale_mle = res.x
    save_samples_mle[i, 0] = c_mle
    save_samples_mle[i, 1] = loc_mle
    save_samples_mle[i, 2] = scale_mle
    
    # Generate samples using MLE parameters
    #p_mle_all_cities[i, :] = genextreme.rvs(c=c_mle, loc=loc_mle, scale=scale_mle, size=sample_number)
    
    # Calculate percentiles
    #percentiles_all_cities[i, :] = np.percentile(p_mle_all_cities[i, :], percentiles)
    percentiles_all_cities[i, :] = genextreme.ppf(p, c_mle, loc_mle, scale_mle)

# Save results to files
#np.savez(r"D:\IDF bayesian\samples\all_samples_and_p_mle_24h_AORC.npz", saved_samples_24h_mle=save_samples_mle, p_mle_24h=p_mle_all_cities)
percentile_table = pd.DataFrame(percentiles_all_cities, columns=percentiles, index=cities)
percentile_table.index.name = 'City'
percentile_table.to_csv(r'D:\BMM-IDF4Drainage_data_results\Percentile\percentile_table_24h_mle_AORC.csv')

params_table = pd.DataFrame(save_samples_mle, columns=['shape(c)', 'loc', 'scale'], index=cities)
params_table.index.name = 'City'
params_table.to_csv(r'D:\BMM-IDF4Drainage_data_results\Percentile\gev_params_24h_mle_AORC.csv')

In [6]:
# Load the csv table as a DataFrame
df = pd.read_csv(r'D:\BMM-IDF4Drainage_data_results\Percentile\percentile_table_24h_mle_AORC.csv')
df = df.round(1)
# Create a dictionary to map old column names to new column names
col_mapping = {'50': '2', '80': '5', '90': '10', '96': '25', '98': '50', '99': '100'}
# Rename columns using the dictionary
df = df.rename(columns=col_mapping)
# Save the renamed CSV file
df.to_csv(r'D:\BMM-IDF4Drainage_data_results\Percentile\return_levels(mm)_24h_mle_AORC.csv', index=False)
print(df)


              City      2      5     10     25     50    100
0       Birmingham   79.2  107.5  127.0  152.7  172.4  192.6
1          Phoenix   24.7   36.5   45.6   59.0   70.4   83.2
2      Little Rock   77.6  105.6  128.4  163.2  194.1  230.0
3      Los Angeles   57.6   86.9  106.8  132.8  152.6  172.7
4           Denver   34.6   45.6   52.9   62.0   68.8   75.4
5         Hartford   66.6   84.4   96.4  111.8  123.5  135.1
6       Wilmington   63.5   86.0  104.0  131.2  155.1  182.5
7            Miami   77.4  115.4  150.0  208.5  265.9  338.4
8          Atlanta   68.1   89.1  105.1  128.1  147.4  168.6
9            Boise   21.7   28.0   32.1   37.3   41.2   44.9
10         Chicago   54.1   72.5   85.9  104.7  120.0  136.5
11    Indianapolis   56.5   72.8   83.9   98.2  109.0  119.9
12      Des Moines   52.4   65.2   75.0   89.2  101.1  114.3
13         Wichita   62.8   85.3  102.1  125.6  145.0  166.1
14      Louisville   64.8   88.8  105.0  126.0  141.8  157.8
15     New Orleans  105.

## CMIP6

In [13]:
# Define GEV log-likelihood function
def gev_loglik(theta, data):
    c, loc, scale = theta
    gev = genextreme(c=c, loc=loc, scale=scale)
    return -np.sum(np.log(gev.pdf(data)))

# Load dataframes for each city

sample_number = 20000

T = np.array([2, 5, 10, 25, 50, 100])
p = 1 - 1 / T
for scen in ['126_2025_2064','126_2065_2100','370_2025_2064','370_2065_2100','585_2025_2064','585_2065_2100','585_2025_2100']:
    # Initialize arrays to store results
    save_samples_mle = np.zeros([len(cities),3])
    p_mle_all_cities = np.zeros([len(cities), sample_number])
    percentiles_all_cities = np.zeros([len(cities), 6])
    percentiles = [50, 80, 90, 96, 98, 99]

    # Iterate over all cities
    for i in range(len(cities)):
        # Get 24h data
        data24h = np.array(dataframes_ssp.get(cities[i])[scen])
        
        # Remove NaN values
        data24h = data24h[~np.isnan(data24h)]
        
        # Fit GEV distribution using MLE
        res = minimize(gev_loglik, x0=[0, 0, 1], args=(data24h,), method='Nelder-Mead')
        c_mle, loc_mle, scale_mle = res.x
        save_samples_mle[i, 0] = c_mle
        save_samples_mle[i, 1] = loc_mle
        save_samples_mle[i, 2] = scale_mle
                
        # Calculate percentiles
        #percentiles_all_cities[i, :] = np.percentile(p_mle_all_cities[i, :], percentiles)
        percentiles_all_cities[i, :] = genextreme.ppf(p, c_mle, loc_mle, scale_mle)

    # Save results to files
    #np.savez(r"D:\IDF bayesian\samples\all_samples_and_p_mle_24h_AORC.npz", saved_samples_24h_mle=save_samples_mle, p_mle_24h=p_mle_all_cities)
    percentile_table = pd.DataFrame(percentiles_all_cities, columns=percentiles, index=cities)
    percentile_table.index.name = 'City'
    percentile_table.to_csv(fr'D:\BMM-IDF4Drainage_data_results\Percentile\percentile_table_24h_mle_CMIP6_{scen}.csv')

    params_table = pd.DataFrame(save_samples_mle, columns=['shape(c)', 'loc', 'scale'], index=cities)
    params_table.index.name = 'City'
    params_table.to_csv(fr'D:\BMM-IDF4Drainage_data_results\Percentile\gev_params_24h_mle_CMIP6_{scen}.csv')

In [14]:
for scen in ['126_2025_2064','126_2065_2100','370_2025_2064','370_2065_2100','585_2025_2064','585_2065_2100','585_2025_2100']:
    # Load the csv table as a DataFrame
    df = pd.read_csv(fr'D:\BMM-IDF4Drainage_data_results\Percentile\percentile_table_24h_mle_CMIP6_{scen}.csv')
    df = df.round(1)
    # Create a dictionary to map old column names to new column names
    col_mapping = {'50': '2', '80': '5', '90': '10', '96': '25', '98': '50', '99': '100'}
    # Rename columns using the dictionary
    df = df.rename(columns=col_mapping)
    # Save the renamed CSV file
    df.to_csv(fr'D:\BMM-IDF4Drainage_data_results\Percentile\return_levels(mm)_24h_mle_CMIP6_{scen}.csv', index=False)

In [20]:
#Getting the current date and time
current_datetime = datetime.now()

# Printing the date and time
print("Date and Time of the Notebook Analysis:", current_datetime)

Date and Time of the Notebook Analysis: 2025-10-23 11:55:41.699513


In [21]:
%load_ext watermark

# Print the Python version and some dependencies
%watermark -v -m -p numpy,pandas,matplotlib,pymc3,scipy,seaborn,arviz,os,theano,warnings,tqdm,watermark


The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark
Python implementation: CPython
Python version       : 3.9.19
IPython version      : 8.18.1

numpy     : 1.22.1
pandas    : 2.0.3
matplotlib: 3.8.4
pymc3     : 3.11.6
scipy     : 1.7.3
seaborn   : 0.13.2
arviz     : 0.12.1
os        : unknown
theano    : 1.1.2
warnings  : unknown
tqdm      : 4.66.4
watermark : 2.4.3

Compiler    : MSC v.1929 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
CPU cores   : 24
Architecture: 64bit

